# Image Segmentation with SAM (Segment Anything Model)

In this exercise, we delve into the world of image segmentation using the advanced SAM (Segment Anything Model). We will explore how to employ this model for segmenting specific parts of an image, a crucial step in various computer vision tasks. By the end, we'll segment an image of a butterfly by providing SAM with a bounding box. We'll see it segment the butterfly with extreme precision.

## Setup

First, let's import the necessary libraries. We use OpenCV for image processing, NumPy for numerical operations, and Matplotlib for visualization.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor

## Helper Functions

To effectively visualize our segmentation results, we define some helper functions. These functions will assist us in overlaying segmentation masks and drawing bounding boxes on our images.

In [ ]:
def show_mask(mask, ax):
    color = np.array([30/255, 144/255, 255/255, 0.6]) # An opaque blue color we'll use to indicate the mask
    
    # TODO: Implement the function to overlay a color mask on the image. 
    # Hint: Using the color array, reshape the mask, and multiply with the color for overlay.
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_box(box, ax):
    # TODO: Complete this function to draw a bounding box on the image.
    # Hint: Use plt.Rectangle to draw the box.
    
    box = np.array(box).reshape(-1)
    x0, y0, x1, y1 = box[:4]
    w, h = x1 - x0, y1 - y0
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='red', facecolor='none', lw=2))

## Loading and Preparing the Model

We will now load a pre-trained SAM model. SAM models are potent for various segmentation tasks and come with pre-trained weights.

In [ ]:
# Load the SAM model with pre-trained weights
from pathlib import Path
import urllib.request
import torch

model_type = "vit_l"
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_dir = Path(r"c:\GitHub\PyTorchTensor\checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Standard SAM checkpoint for segment_anything
sam_checkpoint = ckpt_dir / "sam_vit_l_0b3195.pth"
if not sam_checkpoint.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",
        sam_checkpoint,
    )

sam = sam_model_registry[model_type](checkpoint=str(sam_checkpoint))
sam.to(device=device)
predictor = SamPredictor(sam)

## Image Preparation

Next, we load an image for our segmentation task. We convert the image to the RGB color space, as the SAM model expects input in this format.

In [ ]:
# Load and preprocess an image for segmentation

# ...existing code...
from pathlib import Path
import urllib.request
import cv2

assets = Path(r"c:\GitHub\PyTorchTensor\Examples\05-computer-vision-gen-ai\assets")
assets.mkdir(parents=True, exist_ok=True)
img_path = assets / "butterfly.jpg"

urls = [
    "https://upload.wikimedia.org/wikipedia/commons/4/4c/Monarch_Butterfly_Danaus_plexippus.jpg",
    "https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/truck.jpg",  # fallback
]

headers = {"User-Agent": "Mozilla/5.0"}

last_err = None
for url in urls:
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=30) as r, open(img_path, "wb") as f:
            f.write(r.read())
        print("Downloaded:", url)
        break
    except Exception as e:
        last_err = e
else:
    raise RuntimeError(f"All downloads failed. Last error: {last_err}")

image_bgr = cv2.imread(str(img_path))
if image_bgr is None:
    raise FileNotFoundError(f"Unreadable image: {img_path}")

image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print("Loaded:", img_path)
# ...existing code...

## Conducting and Visualizing Segmentation

Let's perform the actual segmentation on our image. We'll define the input parameters for our segmentation task and apply the SAM model.

In [ ]:
# Define parameters for segmentation
input_box = np.array([[306, 132, 925, 893]]) # Do not change

# Segmentation using SAM
predictor.set_image(image)

# TODO: Use the predictor to perform segmentation. Pay attention to the parameters and how they might affect the segmentation output.
# Hint: You need to call predictor.predict() 
# Hint: Choose from below arguments:
"""
      point_coords (np.ndarray or None): A Nx2 array of point prompts to the
        model. Each point is in (X,Y) in pixels.
      point_labels (np.ndarray or None): A length N array of labels for the
        point prompts. 1 indicates a foreground point and 0 indicates a
        background point.
      box (np.ndarray or None): A length 4 array given a box prompt to the
        model, in XYXY format.
      mask_input (np.ndarray): A low resolution mask input to the model, typically
        coming from a previous prediction iteration. Has form 1xHxW, where
        for SAM, H=W=256.
"""
# Hint: Use the argument "hq_token_only = True" for higher accuracy

masks, scores, logits = predictor.predict(
    point_coords=None,
    point_labels=None,
    box=input_box[0],          # XYXY
    multimask_output=True,
)

Next, let's write the segmentation visualization logic.

In [ ]:
def show_res(masks, scores, input_box, image):
    # TODO: Iterate over the masks and scores, use the visualization functions to display the results.
    # Hint: First display the image, then display mask and box on top
    # Hint: Use plt.imshow(image)
    # Hint: Use show_box and show_mask
    
    # TODO: Print the final score

    for i, (mask, score) in enumerate(zip(masks, scores), start=1):
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(image)
        show_mask(mask, ax)
        show_box(input_box, ax)
        ax.set_title(f"Mask {i} - Score: {score:.4f}")
        ax.axis("off")
        plt.show()

        best_idx = int(np.argmax(scores))
    print(f"Final score (best mask): {scores[best_idx]:.4f}")

In [ ]:
# Visualize the segmentation results
show_res(masks, scores, input_box, image)